# Decision Tree Classification Practical — Simple Exam Notebook

**Scenario:** Loan Approval Prediction  
**Dataset:** `loan.csv`  
**Target:** `loan_status` (`Approved` / `Rejected`)

## What we will do
1. Load and inspect the dataset.
2. Clean column names and handle categorical predictors with `OneHotEncoder`.
3. Separate predictors (`X`) and target (`y`).
4. Build Decision Tree without PCA and tune using 5-fold cross-validation.
5. Build Decision Tree with PCA and tune similarly.
6. Compare performance and run paired t-test (`ttest_rel`).

**Important:** Keep `loan.csv` in the same directory as this notebook.


## 1. Import the tools

Tools from the cheatsheet: pandas, numpy, matplotlib, seaborn, SimpleImputer, StandardScaler, OneHotEncoder, ColumnTransformer, Pipeline, DecisionTreeClassifier, PCA, GridSearchCV, StratifiedKFold, and ttest_rel.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from scipy.stats import ttest_rel


## 2. Load the dataset and inspect it


In [ ]:
df = pd.read_csv("loan.csv")
df.columns = df.columns.str.strip()

print("First five rows:")
display(df.head())

print("Dataset shape:", df.shape)
print("\nColumn information:")
df.info()

print("\nMissing values per column:")
print(df.isnull().sum())


## 3. Explore target and clean features


In [ ]:
target_col = "loan_status"

df[target_col] = df[target_col].str.strip()
print("Class counts:")
print(df[target_col].value_counts())

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=target_col)
plt.title("Loan Approval Distribution")
plt.show()

# Drop loan_id if present
if "loan_id" in df.columns:
    df.drop(columns=["loan_id"], inplace=True)


## 4. Separate features and target, and define ColumnTransformer


In [ ]:
X = df.drop(columns=[target_col])
y = np.where(df[target_col] == "Approved", 1, 0)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric columns:", num_cols)
print("Categorical columns:", cat_cols)

preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])


## 5. Build Decision Tree without PCA and tune with GridSearchCV


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_no_pca = Pipeline([
    ("prep", preprocessor),
    ("dt", DecisionTreeClassifier(random_state=42))
])

param_grid_no_pca = {
    "dt__max_depth": [3, 5, 7, 10],
    "dt__criterion": ["gini", "entropy"],
    "dt__min_samples_split": [2, 5, 10]
}

grid_no_pca = GridSearchCV(
    estimator=pipe_no_pca,
    param_grid=param_grid_no_pca,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_no_pca.fit(X, y)

print("Best Parameters (No PCA):", grid_no_pca.best_params_)
print("Best CV Accuracy (No PCA):", round(grid_no_pca.best_score_, 4))


## 6. Build Decision Tree with PCA and tune with GridSearchCV


In [ ]:
pipe_pca = Pipeline([
    ("prep", preprocessor),
    ("pca", PCA(n_components=0.95, random_state=42)),
    ("dt", DecisionTreeClassifier(random_state=42))
])

param_grid_pca = {
    "dt__max_depth": [3, 5, 7, 10],
    "dt__criterion": ["gini", "entropy"],
    "dt__min_samples_split": [2, 5, 10]
}

grid_pca = GridSearchCV(
    estimator=pipe_pca,
    param_grid=param_grid_pca,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid_pca.fit(X, y)

print("Best Parameters (With PCA):", grid_pca.best_params_)
print("Best CV Accuracy (With PCA):", round(grid_pca.best_score_, 4))


## 7. Compare Models and Paired t-test


In [ ]:
best_no_pca = grid_no_pca.best_estimator_
best_pca = grid_pca.best_estimator_

scores_no_pca = cross_val_score(best_no_pca, X, y, cv=cv, scoring="accuracy")
scores_pca = cross_val_score(best_pca, X, y, cv=cv, scoring="accuracy")

print("Fold Scores (No PCA):", np.round(scores_no_pca, 4))
print("Mean Accuracy (No PCA):", round(scores_no_pca.mean(), 4))
print("\nFold Scores (With PCA):", np.round(scores_pca, 4))
print("Mean Accuracy (With PCA):", round(scores_pca.mean(), 4))

t_stat, p_val = ttest_rel(scores_no_pca, scores_pca)
print(f"\nPaired t-test: t-statistic = {t_stat:.4f}, p-value = {p_val:.4f}")

if p_val < 0.05:
    print("Significant difference between models (p < 0.05)")
else:
    print("No significant difference between models (p >= 0.05)")


## 8. Confusion Matrix and Classification Report


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

best_no_pca.fit(X_train, y_train)
y_pred = best_no_pca.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
